[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/gw-lvk-python-course/blob/main/Cosmology/Lesson_03_Parameter_Estimation_on_Real_Events.ipynb)

# Lesson 3 — Parameter Estimation on Real Events

**Cosmology Module · LVK Python Course**

In this notebook we apply the Bayesian parameter-estimation machinery from Lesson 2 to **real gravitational-wave events** from the LIGO–Virgo–KAGRA (LVK) catalog.  
We will work with two representative events:

* **GW170817** — the first observed binary neutron star (BNS) merger, a landmark multi-messenger event.
* **GW150914** — the first detected binary black hole (BBH) merger, which opened the era of gravitational-wave astronomy.

The workflow follows a realistic research pipeline:

1. Fetch open strain data from the Gravitational-Wave Open Science Center (**GWOSC**).
2. Configure and run **Bilby** parameter-estimation jobs.
3. Extract and visualise **marginal posteriors** for luminosity distance $d_L$, chirp mass $\mathcal{M}$, and effective spin $\chi_{\rm eff}$.
4. Compare the recovered posteriors against **published LVK samples** from the GWTC catalog.

---

## Learning goals

By the end of this lesson you should be able to:

1. Download and preprocess real GWOSC strain data using `gwpy` and `gwosc`.
2. Explain the role of **PSD estimation** and data conditioning in real PE runs.
3. Set up a Bilby PE job targeting $d_L$, $\mathcal{M}$, $\chi_{\rm eff}$, and other CBC parameters.
4. Interpret and visualise corner plots of the recovered posterior.
5. Download published LVK posterior samples from the GWTC catalog.
6. Perform a quantitative comparison between your Bilby results and the official LVK posteriors.
7. Discuss sources of systematic error and the effect of prior choices on the recovered posteriors.


## Table of contents

1. [Physical background: from strain to parameters](#sec1)
2. [Key parameters: $d_L$, $\mathcal{M}$, and $\chi_{\rm eff}$](#sec2)
3. [The GWOSC data ecosystem](#sec3)
4. [Data conditioning: PSD, whitening, and windowing](#sec4)
5. [Setting up a Bilby run on GWOSC data](#sec5)
6. [Case study: GW170817 (Binary Neutron Star)](#sec6)
7. [Case study: GW150914 (Binary Black Hole)](#sec7)
8. [Comparing with the GWTC published posteriors](#sec8)
9. [Systematic effects and prior sensitivity](#sec9)
10. [Student exercises](#sec10)
11. [References](#sec11)


In [ ]:
# ---------------------------------------------------------------------------
# Install required packages (run once per Colab session or fresh environment)
# ---------------------------------------------------------------------------
import subprocess, sys

pkgs = [
    "bilby",
    "bilby_pipe",
    "dynesty",
    "lalsuite",
    "python-ligo-lw",
    "corner",
    "matplotlib",
    "numpy",
    "scipy",
    "astropy",
    "gwpy",
    "gwosc",
    "pesummary",
    "h5py",
    "requests",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Installation complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Standard imports
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import corner
import scipy.stats as stats

# GW-specific
import bilby
from bilby.core.prior import Uniform, PowerLaw, Sine, Cosine
from bilby.gw.prior import BBHPriorDict, BNSPriorDict
import gwpy
from gwpy.timeseries import TimeSeries
from gwpy.frequencyseries import FrequencySeries
from gwosc.datasets import event_gps
from gwosc import datasets as gwosc_datasets

# Cosmology / utility
from astropy.cosmology import Planck18
import astropy.units as u

# Notebook display
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.4,
})
bilby.core.utils.logger.setLevel("WARNING")
print(f"bilby version : {bilby.__version__}")
print(f"gwpy  version : {gwpy.__version__}")

---
<a id='sec1'></a>
## 1 · Physical background: from strain to parameters

### 1.1 The GW signal model

A gravitational-wave detector measures a dimensionless **strain** $h(t)$, which is a linear combination of the two polarisations:

$$
h(t) = F_+(\alpha,\delta,\psi)\,h_+(t;\boldsymbol{\theta}) + F_\times(\alpha,\delta,\psi)\,h_\times(t;\boldsymbol{\theta}),
$$

where $F_+$ and $F_\times$ are the **antenna pattern functions** (determined by source sky position $\alpha,\delta$ and polarisation angle $\psi$), and $(h_+, h_\times)$ are the two GW polarisation amplitudes predicted by GR from source parameters $\boldsymbol{\theta}$.

### 1.2 CBC parameter space

For a compact binary coalescence (CBC) the full parameter vector has **15 dimensions** (for a quasi-circular, non-precessing binary):

| Category | Parameters |
|---|---|
| **Intrinsic** | $m_1, m_2$ (or equivalently $\mathcal{M}, q$); $\chi_{1z}, \chi_{2z}$ (aligned spins); tidal deformabilities $\Lambda_1,\Lambda_2$ (BNS only) |
| **Extrinsic** | $d_L$, $\iota$ (inclination), $\alpha$ (RA), $\delta$ (Dec), $\psi$ (polarisation), $t_c$ (coalescence time), $\phi_c$ (phase at coalescence) |

### 1.3 The matched-filter SNR

The signal-to-noise ratio of a template $\hat{h}$ against data $d$ is

$$
\rho = \frac{\langle d, \hat{h} \rangle}{\sqrt{\langle \hat{h}, \hat{h} \rangle}},
\quad
\langle a, b \rangle = 4\,\text{Re}\int_0^\infty \frac{\tilde{a}(f)\tilde{b}^*(f)}{S_n(f)}\,df,
$$

where $S_n(f)$ is the one-sided **power spectral density** (PSD) of the detector noise.  The log-likelihood is:

$$
\ln \mathcal{L}(d|\boldsymbol{\theta}) = -\frac{1}{2}\langle d - h(\boldsymbol{\theta}), d - h(\boldsymbol{\theta}) \rangle + \text{const}.
$$

### 1.4 The role of the PSD

Accurate noise characterisation is critical.  The PSD $S_n(f)$ is estimated from **off-source data** (typically 4–512 seconds around the event).  Errors in the PSD directly bias parameter estimation.  In practice:

* A **median PSD** is computed from many Welch-averaged segments.
* Data are **whitened** ($\tilde{d}/\sqrt{S_n}$) before likelihood evaluation.
* A **Tukey window** is applied to reduce spectral leakage.


---
<a id='sec2'></a>
## 2 · Key parameters: $d_L$, $\mathcal{M}$, and $\chi_{\rm eff}$

We focus on three parameters of special cosmological and astrophysical importance.

### 2.1 Luminosity distance $d_L$

The GW strain amplitude scales as $h \propto \mathcal{M}^{5/3}/d_L$.  Because GR predicts the amplitude from the intrinsic parameters, $d_L$ is directly measurable from the waveform.  This is the cornerstone of the **standard-siren** method for measuring $H_0$.

The prior on $d_L$ is typically chosen to be **uniform in comoving volume**, which in the Euclidean limit gives $p(d_L) \propto d_L^2$.

### 2.2 Chirp mass $\mathcal{M}$

The chirp mass is the best-constrained mass parameter because it governs the leading-order GW phase evolution:

$$
\mathcal{M} = \frac{(m_1 m_2)^{3/5}}{(m_1+m_2)^{1/5}}.
$$

For a detector at redshift $z$, we measure the **redshifted chirp mass**
$\mathcal{M}^{\rm det} = (1+z)\,\mathcal{M}^{\rm source}$,
so a cosmological model is needed to convert between the two.

### 2.3 Effective inspiral spin $\chi_{\rm eff}$

The most measurable spin combination is the **mass-weighted aligned spin**:

$$
\chi_{\rm eff} = \frac{m_1 \chi_{1z} + m_2 \chi_{2z}}{m_1 + m_2} \in [-1, 1].
$$

It characterises whether the orbit is spun up or down by the components’ spins and has important implications for binary formation channels.


In [ ]:
# ---------------------------------------------------------------------------
# Quick illustration: chirp mass and effective spin relationships
# ---------------------------------------------------------------------------

def chirp_mass(m1, m2):
    """Compute chirp mass from component masses."""
    return (m1 * m2)**0.6 / (m1 + m2)**0.2

def chi_eff(m1, m2, chi1z, chi2z):
    """Compute effective inspiral spin."""
    return (m1 * chi1z + m2 * chi2z) / (m1 + m2)

# GW170817 approximate parameters
m1_bns, m2_bns = 1.46, 1.27  # solar masses
Mc_bns = chirp_mass(m1_bns, m2_bns)
print(f"GW170817: M1={m1_bns} M_sun, M2={m2_bns} M_sun")
print(f"          Chirp mass Mc = {Mc_bns:.4f} M_sun")
print(f"          Mass ratio   q = {m2_bns/m1_bns:.3f}")
print()

# GW150914 approximate parameters
m1_bbh, m2_bbh = 36.2, 29.1  # solar masses
Mc_bbh = chirp_mass(m1_bbh, m2_bbh)
print(f"GW150914: M1={m1_bbh} M_sun, M2={m2_bbh} M_sun")
print(f"          Chirp mass Mc = {Mc_bbh:.2f} M_sun")
print(f"          Mass ratio   q = {m2_bbh/m1_bbh:.3f}")
print()

# Illustrate chi_eff for a range of spin combinations
chi1_range = np.linspace(-1, 1, 200)
chi2_range = np.linspace(-1, 1, 200)
CHI1, CHI2 = np.meshgrid(chi1_range, chi2_range)
CHIEFF = chi_eff(m1_bbh, m2_bbh, CHI1, CHI2)

fig, ax = plt.subplots(figsize=(6, 5))
cp = ax.contourf(CHI1, CHI2, CHIEFF, levels=20, cmap="RdBu_r")
plt.colorbar(cp, ax=ax, label=r"$\chi_{\rm eff}$")
ax.set_xlabel(r"$\chi_{1z}$")
ax.set_ylabel(r"$\chi_{2z}$")
ax.set_title(r"Effective spin $\chi_{\rm eff}$ for GW150914-like masses")
plt.tight_layout()
plt.show()

---
<a id='sec3'></a>
## 3 · The GWOSC data ecosystem

The **Gravitational-Wave Open Science Center** (GWOSC, [gwosc.org](https://gwosc.org)) provides:

| Resource | Description |
|---|---|
| **Strain data** | Calibrated $h(t)$ at 4096 Hz or 16384 Hz for all public events |
| **PSD files** | Estimated noise curves for each observing run |
| **Posterior samples** | Official LVK PE results for GWTC-1, GWTC-2, GWTC-3 |
| **Event catalog** | Metadata (GPS time, SNR, FAR, source classification) |

### 3.1 Python access via `gwosc` and `gwpy`

* `gwosc` — lightweight API for querying event metadata and data URLs.
* `gwpy` — full-featured library for reading, processing, and plotting GW time-series and frequency-series data.

### 3.2 Event GPS times

Each event has a GPS trigger time used to locate the relevant strain data.


In [ ]:
# ---------------------------------------------------------------------------
# Query event metadata from GWOSC
# ---------------------------------------------------------------------------
from gwosc.datasets import event_gps, event_detectors
from gwosc import datasets as gwosc_datasets

events_of_interest = ["GW170817", "GW150914", "GW190814", "GW151226"]

print(f"{'Event':<15} {'GPS time':>18} {'Detectors'}")
print("-" * 55)
for ev in events_of_interest:
    try:
        gps  = event_gps(ev)
        dets = event_detectors(ev)
        print(f"{ev:<15} {gps:>18.2f} {', '.join(dets)}")
    except Exception as exc:
        print(f"{ev:<15} [query failed: {exc}]")

In [ ]:
# ---------------------------------------------------------------------------
# Fetch and plot open strain data around GW170817
# ---------------------------------------------------------------------------
GW170817_GPS = 1187008882.43
SEGMENT_DURATION = 32   # seconds around merger

# Fetch Hanford (H1) data
t_start_h1 = GW170817_GPS - SEGMENT_DURATION / 2
t_end_h1   = GW170817_GPS + SEGMENT_DURATION / 2

print("Fetching H1 strain data for GW170817 ...")
try:
    h1_strain = TimeSeries.fetch_open_data(
        "H1", t_start_h1, t_end_h1, sample_rate=4096, cache=True
    )
    print(f"  H1 data loaded: {len(h1_strain)} samples at {h1_strain.sample_rate} Hz")
    data_available = True
except Exception as exc:
    print(f"  Could not fetch data: {exc}")
    print("  Using synthetic noise for illustration.")
    data_available = False
    rng = np.random.default_rng(42)
    times = np.linspace(t_start_h1, t_end_h1, SEGMENT_DURATION * 4096)
    h1_strain = TimeSeries(
        rng.normal(0, 1e-21, len(times)),
        t0=t_start_h1, dt=1/4096, unit=""
    )

# Plot raw strain
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(h1_strain.times.value - GW170817_GPS, h1_strain.value, lw=0.5, color="steelblue")
ax.axvline(0, color="red", lw=1.5, ls="--", label="Merger time")
ax.set_xlabel("Time relative to merger (s)")
ax.set_ylabel(r"Strain $h(t)$")
ax.set_title("LIGO-Hanford raw strain around GW170817")
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='sec4'></a>
## 4 · Data conditioning: PSD, whitening, and windowing

Before running parameter estimation, the raw strain must be **conditioned**:

1. **Bandpass filter** — retain astrophysically relevant frequencies (typically 20–2048 Hz).
2. **PSD estimation** — compute the noise power spectral density using Welch’s method on off-source segments.
3. **Whitening** — divide the Fourier-domain data by $\sqrt{S_n(f)}$ to make noise flat.
4. **Windowing** — apply a Tukey (cosine-taper) window to suppress edge effects.

### 4.1 Power spectral density

Welch’s method averages periodograms from overlapping segments:

$$
\hat{S}_n(f) = \frac{1}{K} \sum_{k=1}^{K} |\tilde{d}_k(f)|^2,
$$

where each segment is windowed before FFT to reduce spectral leakage.  The LVK uses a **median-mean** combination of segment periodograms to suppress non-Gaussian artefacts.

### 4.2 Whitened Q-transform (spectrogram)

A useful visual diagnostic is the **Q-transform** spectrogram, which shows time-frequency power.  After whitening, a real GW signal appears as a characteristic **chirp**: a track sweeping from low to high frequency as the binary inspirals and merges.


In [ ]:
# ---------------------------------------------------------------------------
# Estimate and plot the PSD for GW170817 (H1)
# ---------------------------------------------------------------------------
FFT_LENGTH  = 4   # seconds per Welch segment
OVERLAP     = 2   # seconds
SAMPLE_RATE = 4096

if data_available:
    psd_h1 = h1_strain.psd(fftlength=FFT_LENGTH, overlap=OVERLAP, method="median")
else:
    # Synthetic PSD: approximate aLIGO design curve
    freqs = np.linspace(20, 2048, 2000)
    # Rough model: power law with noise bumps
    psd_vals = 1e-46 * (freqs / 200)**(-4) + 3e-48 * (freqs / 200)**2 + 1e-47
    psd_h1 = FrequencySeries(psd_vals, frequencies=freqs, unit="")

fig, ax = plt.subplots(figsize=(9, 4))
ax.loglog(psd_h1.frequencies.value, np.sqrt(psd_h1.value),
          color="steelblue", lw=1.5, label="H1 estimated PSD (GW170817)")
ax.set_xlim(20, 2000)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel(r"Strain noise ASD $[1/\sqrt{\rm Hz}]$")
ax.set_title("LIGO-Hanford noise ASD around GW170817")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Whiten and bandpass the strain, then compute Q-transform spectrogram
# ---------------------------------------------------------------------------
if data_available:
    # Whiten using estimated PSD
    h1_white = h1_strain.whiten(fftlength=FFT_LENGTH, overlap=OVERLAP)
    h1_bp    = h1_white.bandpass(50, 2000)   # keep 50–2000 Hz

    # Q-transform around merger
    dt_qtransform = 0.5   # zoom in on the last 0.5 s
    qscan = h1_strain.crop(
        GW170817_GPS - dt_qtransform - 1,
        GW170817_GPS + dt_qtransform
    ).q_transform(outseg=(GW170817_GPS - dt_qtransform,
                           GW170817_GPS + dt_qtransform),
                  frange=(50, 500),
                  qrange=(4, 64))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Whitened time-series
    axes[0].plot(
        h1_bp.times.value - GW170817_GPS,
        h1_bp.value, lw=0.5, color="darkorange"
    )
    axes[0].axvline(0, color="red", lw=1.5, ls="--", label="Merger")
    axes[0].set_xlim(-0.5, 0.1)
    axes[0].set_xlabel("Time relative to merger (s)")
    axes[0].set_ylabel("Whitened strain (normalised)")
    axes[0].set_title("Whitened + bandpassed H1 strain")
    axes[0].legend()

    # Q-transform spectrogram
    plot = qscan.plot(figsize=(7, 4))
    ax2 = plot.gca()
    ax2.set_yscale("log")
    ax2.set_ylim(50, 500)
    ax2.set_title("H1 Q-transform around GW170817 merger")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Frequency (Hz)")
    plot.colorbar(label="Normalised energy")
    plot.show()

    plt.show()
else:
    print("Skipping whitening and Q-transform (data not available).")
    print("In a real session with GWOSC access these cells produce:")
    print("  - Whitened time-series showing the BNS chirp")
    print("  - Q-transform spectrogram with the characteristic upward-sweep")

---
<a id='sec5'></a>
## 5 · Setting up a Bilby run on GWOSC data

### 5.1 Overall workflow

A complete PE run with Bilby on real data involves the following steps:

```
GWOSC data  →  gwpy (fetch & condition)  →  bilby.gw.detector.Interferometer
                                                        ↓
             bilby.gw.likelihood.GravitationalWaveTransient
                       +  prior dictionary
                                ↓
                     dynesty nested sampler
                                ↓
               posterior samples (Result object)
```

### 5.2 Interferometer object

`bilby.gw.detector.Interferometer` encapsulates:
* The strain data array.
* The estimated PSD.
* The detector geometry (location, orientation).
* Antenna-pattern functions $F_+$, $F_\times$.

### 5.3 Computational cost

Running a full 15-dimensional PE job on real data typically takes **hours to days** on a cluster.  For this notebook, we use two strategies:

1. **Reduced-order quadrature (ROQ)** — pre-computed basis that evaluates the likelihood ~100× faster.
2. **Low-latency run** — restrict frequency range and sampler settings for a quick exploratory run.
3. **Published posteriors** — download and analyse the official LVK posterior samples directly (Section 8).

> **Practical note** In production you would submit a `bilby_pipe` DAG to an HTCondor or Slurm cluster.  The cells below run a *lightweight* version that completes in a few minutes by fixing well-constrained parameters at fiducial values and shrinking the prior volume.


In [ ]:
# ---------------------------------------------------------------------------
# Build bilby Interferometer objects from GWOSC data (GW150914 example)
# We use GW150914 for the BBH demo because it has the highest SNR.
# ---------------------------------------------------------------------------
GW150914_GPS = 1126259462.4
DURATION     = 4      # seconds of data around the event
POST_TRIGGER = 2      # how much data lies after the trigger
SAMPLING_FREQ = 2048  # Hz (sufficient for BBH below ~1 kHz)
FLOW         = 20     # Hz low-frequency cutoff

ifo_names = ["H1", "L1"]
print(f"Setting up interferometers for GW150914 (GPS = {GW150914_GPS})")
print(f"Data duration: {DURATION} s at {SAMPLING_FREQ} Hz")

try:
    ifos = bilby.gw.detector.InterferometerList([])
    for ifo_name in ifo_names:
        ifo = bilby.gw.detector.get_empty_interferometer(ifo_name)
        ifo.set_strain_data_from_open_data(
            name=ifo_name,
            trigger_time=GW150914_GPS,
            duration=DURATION,
            sampling_frequency=SAMPLING_FREQ,
            psd_duration=32 * DURATION,
            roll_off=0.2,
        )
        ifos.append(ifo)
        print(f"  {ifo_name}: loaded {ifo.strain_data.n_time_samples} samples, "
              f"PSD from {ifo.power_spectral_density_array.shape[0]} freq bins")
    real_data_loaded = True
except Exception as exc:
    print(f"Could not load real data: {exc}")
    print("Falling back to a simulated injection for demonstration.")
    real_data_loaded = False

In [ ]:
# ---------------------------------------------------------------------------
# Plot the ASDs of the two detectors
# ---------------------------------------------------------------------------
if real_data_loaded:
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["steelblue", "darkorange"]
    for ifo, col in zip(ifos, colors):
        freq   = ifo.strain_data.frequency_array
        psd    = ifo.power_spectral_density_array
        mask   = (freq >= FLOW) & (freq <= 1024)
        ax.loglog(freq[mask], np.sqrt(psd[mask]), color=col, lw=1.5, label=ifo.name)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel(r"ASD $[1/\sqrt{\rm Hz}]$")
    ax.set_title("Detector noise ASDs around GW150914")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("(Skipping ASD plot — real data not loaded)")

---
<a id='sec6'></a>
## 6 · Case study: GW170817 (Binary Neutron Star)

GW170817 was detected on 2017 August 17 by the Advanced LIGO and Virgo detectors.  It is the **first confirmed BNS merger** observed in gravitational waves and has an electromagnetic counterpart (kilonova AT2017gfo / GRB 170817A).  Key published properties:

| Parameter | LVK median (90% CI) |
|---|---|
| Chirp mass $\mathcal{M}^{\rm det}$ | $1.1977^{+0.0004}_{-0.0003}\,M_\odot$ |
| Total mass $M$ | $2.74^{+0.04}_{-0.01}\,M_\odot$ |
| Luminosity distance $d_L$ | $40^{+8}_{-14}\,\rm Mpc$ |
| $\chi_{\rm eff}$ | $0.00^{+0.02}_{-0.01}$ (low-spin prior) |
| Host galaxy | NGC 4993, $z=0.009$ |

### 6.1 Why is GW170817 special for PE?

* **High SNR** ($\rho_{\rm net} \approx 32$): posteriors are tight.
* **Long in-band duration** (~100 s from 20 Hz): superb chirp-mass measurement.
* **Known host galaxy** $\Rightarrow$ $z$ known $\Rightarrow$ $H_0$ measurement.
* **Tidal effects** encoded in high-frequency phase: constrains neutron-star equation of state.


In [ ]:
# ---------------------------------------------------------------------------
# GW170817: Prior setup
# For a real BNS run the data segment would be ~128 s and the sampler
# would run for hours.  Here we build the prior dict to illustrate the
# choices, and will load official posteriors in Section 8.
# ---------------------------------------------------------------------------

# Start from the built-in BNS prior
prior_bns = bilby.gw.prior.BNSPriorDict()

# Override specific priors based on known event information
# Chirp mass: very tight for high-SNR BNS events
prior_bns["chirp_mass"] = bilby.core.prior.Uniform(
    name="chirp_mass", minimum=1.18, maximum=1.21,
    latex_label=r"$\mathcal{M}\;[M_\odot]$"
)
# Mass ratio
prior_bns["mass_ratio"] = bilby.core.prior.Uniform(
    name="mass_ratio", minimum=0.7, maximum=1.0,
    latex_label="$q$"
)
# Luminosity distance (low-z event in Virgo cluster)
prior_bns["luminosity_distance"] = bilby.core.prior.PowerLaw(
    alpha=2, name="luminosity_distance", minimum=10, maximum=200,
    unit="Mpc", latex_label=r"$d_L\;[{\rm Mpc}]$"
)
# Low-spin prior: consistent with slow NS spins
prior_bns["a_1"] = bilby.core.prior.Uniform(minimum=0, maximum=0.05, name="a_1")
prior_bns["a_2"] = bilby.core.prior.Uniform(minimum=0, maximum=0.05, name="a_2")

print("GW170817 BNS prior dictionary:")
for name, prior in prior_bns.items():
    print(f"  {name:<30s}: {prior}")

In [ ]:
# ---------------------------------------------------------------------------
# Simulate representative GW170817-like posterior samples
# (These mimic the distributions published in Abbott et al. 2019, PRL)
# In Section 8 we load and compare with the real published samples.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(2024)
N   = 5000

# Chirp mass: very tight, ~0.3 ms uncertainty
Mc_bns_samples  = rng.normal(1.1977, 0.0003, N)

# Luminosity distance: strongly degenerate with inclination
dL_bns_samples  = rng.normal(40, 8, N)
dL_bns_samples  = np.clip(dL_bns_samples, 10, 200)

# Effective spin: near zero (low-spin prior)
chi_eff_bns     = rng.normal(0.00, 0.02, N)
chi_eff_bns     = np.clip(chi_eff_bns, -0.05, 0.05)

# Pack into a dict
bns_posterior = {
    "chirp_mass"          : Mc_bns_samples,
    "luminosity_distance" : dL_bns_samples,
    "chi_eff"             : chi_eff_bns,
}

print(f"Mock GW170817 posteriors ({N} samples)")
for k, v in bns_posterior.items():
    lo, med, hi = np.percentile(v, [5, 50, 95])
    print(f"  {k:<25s}: {med:.4f}  (90% CI: [{lo:.4f}, {hi:.4f}])")

In [ ]:
# ---------------------------------------------------------------------------
# Corner plot for GW170817-like posteriors
# ---------------------------------------------------------------------------
labels_bns  = [
    r"$\mathcal{M}\,[M_\odot]$",
    r"$d_L\,[\rm Mpc]$",
    r"$\chi_{\rm eff}$",
]
samples_bns = np.column_stack([
    bns_posterior["chirp_mass"],
    bns_posterior["luminosity_distance"],
    bns_posterior["chi_eff"],
])

fig = corner.corner(
    samples_bns,
    labels=labels_bns,
    quantiles=[0.05, 0.5, 0.95],
    show_titles=True,
    title_kwargs={"fontsize": 11},
    color="steelblue",
    plot_datapoints=False,
    fill_contours=True,
    levels=[0.68, 0.90],
    smooth=1.0,
)
fig.suptitle("GW170817 (BNS) — posterior marginals", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
<a id='sec7'></a>
## 7 · Case study: GW150914 (Binary Black Hole)

GW150914 was detected on 2015 September 14, the **first gravitational-wave event ever observed**.  It is a high-mass BBH with a very short in-band duration (~0.2 s) and a network SNR of ~24.

| Parameter | LVK median (90% CI) |
|---|---|
| Chirp mass $\mathcal{M}^{\rm det}$ | $28.3^{+1.8}_{-1.5}\,M_\odot$ |
| Primary mass $m_1^{\rm source}$ | $35.6^{+4.8}_{-3.0}\,M_\odot$ |
| Secondary mass $m_2^{\rm source}$ | $30.6^{+3.0}_{-4.4}\,M_\odot$ |
| Luminosity distance $d_L$ | $440^{+150}_{-170}\,\rm Mpc$ |
| $\chi_{\rm eff}$ | $-0.01^{+0.12}_{-0.13}$ |

### 7.1 Differences from a BNS run

* **No tidal parameters** (black holes have $\Lambda = 0$ in GR).
* **Short duration** → chirp mass measured less precisely.
* **Higher mass** → merger frequency ~150 Hz, well within LIGO band.
* **Spin prior** typically broadened to $a_{1,2} \in [0, 0.99]$.


In [ ]:
# ---------------------------------------------------------------------------
# GW150914 Bilby run (lightweight version with reduced sampler settings)
# ---------------------------------------------------------------------------
GW150914_GPS      = 1126259462.4
DURATION_BBH      = 4     # s
POST_TRIGGER_BBH  = 2     # s after trigger
SRATE_BBH         = 2048  # Hz
FLOW_BBH          = 20    # Hz

# --- Prior ---
prior_bbh = bilby.gw.prior.BBHPriorDict()
# Narrow the chirp-mass range around the known value
prior_bbh["chirp_mass"] = bilby.core.prior.Uniform(
    minimum=25, maximum=32,
    name="chirp_mass", latex_label=r"$\mathcal{M}\,[M_\odot]$"
)
prior_bbh["mass_ratio"] = bilby.core.prior.Uniform(
    minimum=0.5, maximum=1.0, name="mass_ratio", latex_label="$q$"
)
# Distance prior: uniform in comoving volume ~ d_L^2
prior_bbh["luminosity_distance"] = bilby.core.prior.PowerLaw(
    alpha=2, minimum=100, maximum=2000,
    name="luminosity_distance", unit="Mpc", latex_label=r"$d_L\,[{\rm Mpc}]$"
)

# Fix geocent_time tightly around the trigger
prior_bbh["geocent_time"] = bilby.core.prior.Uniform(
    minimum=GW150914_GPS - 0.1, maximum=GW150914_GPS + 0.1,
    name="geocent_time", latex_label="$t_c$"
)

print("GW150914 BBH prior dictionary:")
for name, prior in prior_bbh.items():
    print(f"  {name:<30s}: {prior}")

In [ ]:
# ---------------------------------------------------------------------------
# Attempt a real lightweight Bilby run on GW150914 GWOSC data
# (This may take 10-30 minutes on a standard CPU.)
# If you want to skip the full run, set RUN_BILBY = False to use mock samples.
# ---------------------------------------------------------------------------
RUN_BILBY = False  # <-- set to True to run the full PE job

if RUN_BILBY and real_data_loaded:
    # Rebuild ifos for GW150914 (4096-sample)
    ifos_bbh = bilby.gw.detector.InterferometerList([])
    for ifo_name in ["H1", "L1"]:
        ifo = bilby.gw.detector.get_empty_interferometer(ifo_name)
        ifo.set_strain_data_from_open_data(
            name=ifo_name,
            trigger_time=GW150914_GPS,
            duration=DURATION_BBH,
            sampling_frequency=SRATE_BBH,
            psd_duration=32 * DURATION_BBH,
            roll_off=0.2,
        )
        ifos_bbh.append(ifo)

    # Waveform generator
    wfg = bilby.gw.WaveformGenerator(
        duration=DURATION_BBH,
        sampling_frequency=SRATE_BBH,
        frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
        waveform_arguments={
            "waveform_approximant": "IMRPhenomD",
            "reference_frequency": 50.0,
            "minimum_frequency": FLOW_BBH,
        },
    )

    # Likelihood
    likelihood_bbh = bilby.gw.likelihood.GravitationalWaveTransient(
        interferometers=ifos_bbh,
        waveform_generator=wfg,
        priors=prior_bbh,
        distance_marginalization=True,
        phase_marginalization=True,
        time_marginalization=True,
    )

    # Sampler
    result_bbh = bilby.run_sampler(
        likelihood=likelihood_bbh,
        priors=prior_bbh,
        sampler="dynesty",
        nlive=500,
        nact=10,
        maxmcmc=5000,
        outdir="bilby_gw150914",
        label="GW150914_quick",
        resume=True,
        clean=False,
    )
    bbh_posterior = result_bbh.posterior
    print("Run complete. Posterior shape:", bbh_posterior.shape)
else:
    print("Skipping Bilby run (RUN_BILBY=False or data unavailable).")
    print("Generating mock GW150914-like posterior samples instead.")

    rng = np.random.default_rng(2025)
    N   = 5000

    # Realistic (but synthetic) GW150914 posteriors
    Mc_bbh_samples  = rng.normal(28.3, 1.5, N)
    dL_bbh_samples  = np.abs(rng.normal(440, 150, N))  # Mpc
    chi_eff_bbh     = rng.normal(-0.01, 0.12, N)
    chi_eff_bbh     = np.clip(chi_eff_bbh, -0.9, 0.9)
    q_bbh           = rng.uniform(0.6, 1.0, N)

    bbh_posterior = {
        "chirp_mass"          : Mc_bbh_samples,
        "luminosity_distance" : dL_bbh_samples,
        "chi_eff"             : chi_eff_bbh,
        "mass_ratio"          : q_bbh,
    }

    print(f"Mock GW150914 posteriors ({N} samples)")
    for k, v in bbh_posterior.items():
        lo, med, hi = np.percentile(v, [5, 50, 95])
        print(f"  {k:<25s}: {med:.3f}  (90% CI: [{lo:.3f}, {hi:.3f}])")

In [ ]:
# ---------------------------------------------------------------------------
# Corner plot for GW150914-like posteriors
# ---------------------------------------------------------------------------
if isinstance(bbh_posterior, dict):
    # Dictionary (mock run)
    samples_bbh = np.column_stack([
        bbh_posterior["chirp_mass"],
        bbh_posterior["luminosity_distance"],
        bbh_posterior["chi_eff"],
    ])
else:
    # DataFrame (real Bilby result)
    samples_bbh = bbh_posterior[["chirp_mass", "luminosity_distance", "chi_eff"]].values

labels_bbh = [
    r"$\mathcal{M}^{\rm det}\,[M_\odot]$",
    r"$d_L\,[{\rm Mpc}]$",
    r"$\chi_{\rm eff}$",
]

fig = corner.corner(
    samples_bbh,
    labels=labels_bbh,
    quantiles=[0.05, 0.5, 0.95],
    show_titles=True,
    title_kwargs={"fontsize": 11},
    color="darkorange",
    plot_datapoints=False,
    fill_contours=True,
    levels=[0.68, 0.90],
    smooth=1.0,
)
fig.suptitle("GW150914 (BBH) — posterior marginals", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
<a id='sec8'></a>
## 8 · Comparing with the GWTC published posteriors

The LVK releases official posterior samples for every catalog event via GWOSC.  These are the gold-standard results computed with state-of-the-art pipelines (`LALInference`, `Bilby/parallel_bilby`, `RIFT`).  They are stored in HDF5 files following the **PESummary** format.

### 8.1 How to access them

```python
# Using PESummary
from pesummary.io import read
samples = read("IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5")
```

Or directly from the GWOSC REST API:
```
https://gwosc.org/eventapi/json/GWTC/
```

### 8.2 Data releases

| Catalog | Events | Reference |
|---|---|---|
| GWTC-1 | O1+O2 (11 events) | Abbott et al. 2019 (PRX) |
| GWTC-2 | O3a (47 events) | Abbott et al. 2021 (PRX) |
| GWTC-2.1 | O3a reanalysis | Abbott et al. 2024 (PRD) |
| GWTC-3 | O3b (90 events) | Abbott et al. 2023 (PRX) |


In [ ]:
# ---------------------------------------------------------------------------
# Download GWTC posterior samples via GWOSC API
# ---------------------------------------------------------------------------
import requests, os, json

GWOSC_EVENT_API = "https://gwosc.org/eventapi/json/GWTC/"

# Query event list
print("Querying GWOSC catalog API ...")
try:
    resp = requests.get(GWOSC_EVENT_API, timeout=30)
    catalog_data = resp.json()
    # Show first few events
    events_list = list(catalog_data.get("events", {}).keys())
    print(f"GWTC catalog contains {len(events_list)} events.")
    print("First 10 events:", events_list[:10])
    api_available = True
except Exception as exc:
    print(f"API query failed: {exc}")
    api_available = False

In [ ]:
# ---------------------------------------------------------------------------
# Fetch published posterior samples for GW150914 and GW170817 via PESummary
# We use the public GWOSC download links from the GWTC data releases.
# ---------------------------------------------------------------------------
import urllib.request

# GWTC-1 posterior samples (publicly available at GWOSC)
GWTC1_URL = (
    "https://dcc.ligo.org/public/0157/P1800370/005/"
    "GW150914_GWTC-1.hdf5"
)
GWTC1_BNS_URL = (
    "https://dcc.ligo.org/public/0157/P1800370/005/"
    "GW170817_GWTC-1.hdf5"
)

os.makedirs("/tmp/gwtc_samples", exist_ok=True)

def try_download(url, dest):
    """Download a file, return True on success."""
    if os.path.exists(dest):
        print(f"  Already cached: {dest}")
        return True
    try:
        print(f"  Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"  Saved to {dest}")
        return True
    except Exception as exc:
        print(f"  Download failed: {exc}")
        return False

bbh_file = "/tmp/gwtc_samples/GW150914_GWTC-1.hdf5"
bns_file = "/tmp/gwtc_samples/GW170817_GWTC-1.hdf5"

bbh_downloaded = try_download(GWTC1_URL, bbh_file)
bns_downloaded = try_download(GWTC1_BNS_URL, bns_file)

In [ ]:
# ---------------------------------------------------------------------------
# Parse GWTC-1 HDF5 posterior samples for GW150914
# ---------------------------------------------------------------------------
import h5py

def load_gwtc1_samples(filepath):
    """
    Load posterior samples from a GWTC-1 HDF5 file.
    Returns a dict of {parameter: array}.
    """
    data = {}
    with h5py.File(filepath, "r") as f:
        # Explore structure
        print("HDF5 top-level keys:", list(f.keys()))
        # GWTC-1 format: /IMRPhenomPv2_posterior or /Overall_posterior
        for group_name in f.keys():
            group = f[group_name]
            if hasattr(group, "keys"):
                print(f"  Group '{group_name}' keys: {list(group.keys())[:8]}")
                # Load parameter arrays
                for param in group.keys():
                    try:
                        arr = group[param][()]
                        if arr.ndim == 1:
                            data[param] = arr
                    except Exception:
                        pass
    return data

published_bbh = None
if bbh_downloaded:
    try:
        published_bbh = load_gwtc1_samples(bbh_file)
        print(f"\nLoaded {len(published_bbh)} parameters, {len(next(iter(published_bbh.values())))} samples")
        print("Available parameters:", list(published_bbh.keys()))
    except Exception as exc:
        print(f"Could not parse file: {exc}")

if published_bbh is None:
    print("Using the mock posteriors generated in Section 7.")
    published_bbh = bbh_posterior

In [ ]:
# ---------------------------------------------------------------------------
# Parse GWTC-1 HDF5 posterior samples for GW170817
# ---------------------------------------------------------------------------
published_bns = None
if bns_downloaded:
    try:
        published_bns = load_gwtc1_samples(bns_file)
        print(f"\nLoaded {len(published_bns)} parameters, {len(next(iter(published_bns.values())))} samples")
        print("Available parameters:", list(published_bns.keys()))
    except Exception as exc:
        print(f"Could not parse file: {exc}")

if published_bns is None:
    print("Using the mock posteriors generated in Section 6.")
    published_bns = bns_posterior

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side comparison of Bilby (mock) vs published (mock) posteriors
# for d_L and chirp mass.
#
# When you run with real data, replace 'our_*' with the Bilby result arrays
# and 'pub_*' with the columns from the GWTC HDF5 files.
# ---------------------------------------------------------------------------

# For illustration we create two slightly different mock distributions
rng2 = np.random.default_rng(9999)

# GW150914 --- "our run" vs "published"
our_dL_bbh  = bbh_posterior["luminosity_distance"] if isinstance(bbh_posterior, dict) \
              else bbh_posterior["luminosity_distance"].values
our_Mc_bbh  = bbh_posterior["chirp_mass"] if isinstance(bbh_posterior, dict) \
              else bbh_posterior["chirp_mass"].values

# Simulate the official result: slightly broader dL, same Mc
pub_dL_bbh  = rng2.normal(440, 160, 5000)
pub_dL_bbh  = np.abs(pub_dL_bbh)
pub_Mc_bbh  = rng2.normal(28.3, 1.6, 5000)

# GW170817 --- "our run" vs "published"
our_dL_bns  = bns_posterior["luminosity_distance"] if isinstance(bns_posterior, dict) \
              else bns_posterior["luminosity_distance"].values
our_Mc_bns  = bns_posterior["chirp_mass"] if isinstance(bns_posterior, dict) \
              else bns_posterior["chirp_mass"].values

pub_dL_bns  = rng2.normal(40, 9, 5000)
pub_dL_bns  = np.clip(pub_dL_bns, 10, 200)
pub_Mc_bns  = rng2.normal(1.1977, 0.00032, 5000)

# ---- Plot ----
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

def overlay_kde(ax, our, pub, xlabel, title, col_our="steelblue", col_pub="tomato", bins=60):
    ax.hist(our, bins=bins, density=True, alpha=0.5, color=col_our, label="This analysis")
    ax.hist(pub, bins=bins, density=True, alpha=0.5, color=col_pub, label="LVK published")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Probability density")
    ax.set_title(title)
    ax.legend(fontsize=9)

overlay_kde(axes[0, 0], our_Mc_bbh, pub_Mc_bbh,
            r"$\mathcal{M}^{\rm det}\,[M_\odot]$",
            "GW150914: Chirp mass",
            col_our="darkorange", col_pub="slategray")

overlay_kde(axes[0, 1], our_dL_bbh, pub_dL_bbh,
            r"$d_L\,[{\rm Mpc}]$",
            "GW150914: Luminosity distance",
            col_our="darkorange", col_pub="slategray")

overlay_kde(axes[1, 0], our_Mc_bns, pub_Mc_bns,
            r"$\mathcal{M}^{\rm det}\,[M_\odot]$",
            "GW170817: Chirp mass",
            col_our="steelblue", col_pub="mediumseagreen")

overlay_kde(axes[1, 1], our_dL_bns, pub_dL_bns,
            r"$d_L\,[{\rm Mpc}]$",
            "GW170817: Luminosity distance",
            col_our="steelblue", col_pub="mediumseagreen")

fig.suptitle(
    "Comparison: this analysis (mock Bilby run) vs. LVK published posteriors\n"
    "(Replace mock samples with real Bilby results for a genuine comparison)",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Quantitative comparison: Jensen-Shannon divergence and CI overlap
# ---------------------------------------------------------------------------
from scipy.spatial.distance import jensenshannon
from scipy.stats import gaussian_kde

def js_divergence_samples(s1, s2, n_bins=100):
    """
    Estimate Jensen-Shannon divergence between two sets of 1-D samples
    using a histogram approximation.
    """
    lo  = min(s1.min(), s2.min())
    hi  = max(s1.max(), s2.max())
    bins = np.linspace(lo, hi, n_bins + 1)
    p, _ = np.histogram(s1, bins=bins, density=True)
    q, _ = np.histogram(s2, bins=bins, density=True)
    p   = p / p.sum()
    q   = q / q.sum()
    return jensenshannon(p, q)

def credible_interval(samples, levels=(0.05, 0.95)):
    return np.percentile(samples, [100*l for l in levels])

print("\n=== GW150914: Quantitative Posterior Comparison ===")
for name, ours, pub in [
    (r"Chirp mass (M_sun)", our_Mc_bbh, pub_Mc_bbh),
    (r"Luminosity distance (Mpc)", our_dL_bbh, pub_dL_bbh),
]:
    js = js_divergence_samples(ours, pub)
    ci_our = credible_interval(ours)
    ci_pub = credible_interval(pub)
    print(f"  {name}")
    print(f"    Our 90% CI     : [{ci_our[0]:.3f}, {ci_our[1]:.3f}]")
    print(f"    Published 90% CI: [{ci_pub[0]:.3f}, {ci_pub[1]:.3f}]")
    print(f"    JS divergence  : {js:.4f}  (0 = identical, 1 = disjoint)")

print("\n=== GW170817: Quantitative Posterior Comparison ===")
for name, ours, pub in [
    (r"Chirp mass (M_sun)", our_Mc_bns, pub_Mc_bns),
    (r"Luminosity distance (Mpc)", our_dL_bns, pub_dL_bns),
]:
    js = js_divergence_samples(ours, pub)
    ci_our = credible_interval(ours)
    ci_pub = credible_interval(pub)
    print(f"  {name}")
    print(f"    Our 90% CI     : [{ci_our[0]:.4f}, {ci_our[1]:.4f}]")
    print(f"    Published 90% CI: [{ci_pub[0]:.4f}, {ci_pub[1]:.4f}]")
    print(f"    JS divergence  : {js:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# chi_eff comparison: GW150914 vs GW170817
# This illustrates the astrophysical contrast between BBH and BNS spins.
# ---------------------------------------------------------------------------

chi_eff_bbh_arr = bbh_posterior["chi_eff"] if isinstance(bbh_posterior, dict) \
                  else bbh_posterior["chi_eff"].values
chi_eff_bns_arr = bns_posterior["chi_eff"] if isinstance(bns_posterior, dict) \
                  else bns_posterior["chi_eff"].values

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(chi_eff_bbh_arr, bins=50, density=True, alpha=0.6,
        color="darkorange", label=r"GW150914 (BBH)")
ax.hist(chi_eff_bns_arr, bins=50, density=True, alpha=0.6,
        color="steelblue", label=r"GW170817 (BNS, low-spin prior)")

ax.axvline(0, color="black", lw=1, ls="--", alpha=0.5)

# Medians
ax.axvline(np.median(chi_eff_bbh_arr), color="darkorange", lw=2, ls="-",
           label=f"BBH median = {np.median(chi_eff_bbh_arr):.3f}")
ax.axvline(np.median(chi_eff_bns_arr), color="steelblue", lw=2, ls="-",
           label=f"BNS median = {np.median(chi_eff_bns_arr):.4f}")

ax.set_xlabel(r"Effective inspiral spin $\chi_{\rm eff}$")
ax.set_ylabel("Probability density")
ax.set_title(r"Marginal $\chi_{\rm eff}$ posteriors: BBH vs BNS")
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 0.5)
plt.tight_layout()
plt.show()

print("Astrophysical interpretation:")
print("  GW150914 (BBH): chi_eff consistent with small values; moderate uncertainty.")
print("  GW170817 (BNS): chi_eff tightly constrained near 0 by the low-spin prior,")
print("                  consistent with slowly rotating old neutron stars.")

---
<a id='sec9'></a>
## 9 · Systematic effects and prior sensitivity

PE results are only as good as our model assumptions.  Below we discuss the most important sources of systematic error.

### 9.1 Waveform model systematics

Different waveform approximants give slightly different posteriors because they model the merger–ringdown at different levels of accuracy:

| Approximant | Type | Best for |
|---|---|---|
| `IMRPhenomD` | Frequency-domain, aligned-spin | Fast BBH runs |
| `IMRPhenomPv2` | Frequency-domain, precessing | BBH with precession |
| `SEOBNRv4_ROM` | Time-domain + ROM, aligned-spin | Accurate BBH |
| `IMRPhenomPv2_NRTidalv2` | Aligned-spin + tidal | BNS |
| `SEOBNRv4T` | Tidal | BNS (heavy) |

### 9.2 Prior sensitivity

At low SNR, the posterior is significantly shaped by the prior.  The most impactful choices are:

* **Distance prior** $p(d_L) \propto d_L^2$ (uniform in comoving volume) vs flat.
* **Spin prior**: isotropic ($a \in [0, 0.99]$) vs aligned only vs low-spin ($a < 0.05$).
* **Mass prior**: uniform in component masses vs uniform in $\mathcal{M}$-$q$.

### 9.3 Calibration uncertainty

Detector calibration errors — amplitude and phase uncertainties in the response function — are accounted for by marginalising over **spline calibration parameters** (typically 10 nodes per IFO).

### 9.4 PSD uncertainty

The PSD is estimated from a finite data segment and is therefore uncertain.  `BayesLine` or `BayesWave` can perform joint PSD + signal inference, but this is computationally expensive.


In [ ]:
# ---------------------------------------------------------------------------
# Illustrate prior sensitivity on d_L (GW150914-like)
# ---------------------------------------------------------------------------
rng3 = np.random.default_rng(1111)
N    = 10_000
d_true = 440   # Mpc (true value)

# Simulate likelihood: Gaussian in d_L centred on the true value
sigma_dL = 150  # Mpc (width reflecting measurement uncertainty)

# Likelihood grid
dL_grid = np.linspace(50, 1500, 2000)
likelihood_vals = np.exp(-0.5 * ((dL_grid - d_true) / sigma_dL)**2)
likelihood_vals /= likelihood_vals.sum()

# Prior 1: Flat in d_L
prior_flat = np.ones_like(dL_grid)
prior_flat /= prior_flat.sum()

# Prior 2: Uniform in comoving volume ~ d_L^2
prior_vol = dL_grid**2
prior_vol /= prior_vol.sum()

# Prior 3: Strongly informative (tight Gaussian)
prior_info = np.exp(-0.5 * ((dL_grid - 350) / 50)**2)
prior_info /= prior_info.sum()

# Compute posteriors
post_flat = likelihood_vals * prior_flat
post_flat /= post_flat.sum()
post_vol  = likelihood_vals * prior_vol
post_vol  /= post_vol.sum()
post_info = likelihood_vals * prior_info
post_info /= post_info.sum()

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, prior, post, pname, col in zip(
    axes,
    [prior_flat, prior_vol, prior_info],
    [post_flat, post_vol, post_info],
    ["Flat in $d_L$", r"Flat in $d_L^2$ (volume)", "Gaussian prior $\mu=350$, $\sigma=50$"],
    ["steelblue", "darkorange", "mediumseagreen"],
):
    prior_scaled = prior / prior.max() * post.max()
    ax.plot(dL_grid, likelihood_vals / likelihood_vals.max() * post.max(),
            color="gray", lw=1.5, ls="--", label="Likelihood (shape)")
    ax.fill_between(dL_grid, prior_scaled, alpha=0.25, color=col, label="Prior")
    ax.plot(dL_grid, post, color=col, lw=2, label="Posterior")
    ax.axvline(d_true, color="red", lw=1.5, ls=":", label=f"True $d_L={d_true}$ Mpc")
    ax.set_xlabel(r"$d_L$ [Mpc]")
    ax.set_title(pname, fontsize=10)
    ax.legend(fontsize=8)
    # Find MAP
    map_dL = dL_grid[np.argmax(post)]
    ax.axvline(map_dL, color=col, lw=1.5, ls="-.", alpha=0.7,
               label=f"MAP = {map_dL:.0f} Mpc")

axes[0].set_ylabel("Normalised density")
fig.suptitle("Prior sensitivity: effect on $d_L$ posterior (GW150914-like)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Key takeaway:")
print("  The volume prior d_L^2 pushes the posterior to larger distances.")
print("  A strong informative prior can significantly dominate the posterior.")
print("  Always check prior vs posterior support to identify prior-dominated regions.")

In [ ]:
# ---------------------------------------------------------------------------
# Waveform model comparison: simulate how different approximants
# lead to slightly different chirp-mass posteriors
# ---------------------------------------------------------------------------
rng4 = np.random.default_rng(2222)
N_wf  = 5000

# True chirp mass
Mc_true_bbh = 28.3

# Different approximants introduce slightly shifted/broader posteriors
approx_results = {
    "IMRPhenomD"        : rng4.normal(Mc_true_bbh - 0.05, 1.4, N_wf),
    "IMRPhenomPv2"      : rng4.normal(Mc_true_bbh + 0.10, 1.6, N_wf),
    "SEOBNRv4_ROM"      : rng4.normal(Mc_true_bbh + 0.00, 1.5, N_wf),
    "SEOBNRv4P"         : rng4.normal(Mc_true_bbh - 0.03, 1.55, N_wf),
}

fig, ax = plt.subplots(figsize=(9, 4))
colors_wf = ["steelblue", "darkorange", "mediumseagreen", "mediumpurple"]

for (label, samples), col in zip(approx_results.items(), colors_wf):
    ax.hist(samples, bins=60, density=True, alpha=0.5, color=col, label=label)
    ax.axvline(np.median(samples), color=col, lw=1.5, ls="--")

ax.axvline(Mc_true_bbh, color="red", lw=2, ls=":", label=f"Injected $\mathcal{{M}}={Mc_true_bbh}$")
ax.set_xlabel(r"$\mathcal{M}^{\rm det}\,[M_\odot]$")
ax.set_ylabel("Probability density")
ax.set_title("Waveform model systematics: GW150914 chirp mass")
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

print("Systematics summary:")
for label, samples in approx_results.items():
    lo, med, hi = np.percentile(samples, [5, 50, 95])
    print(f"  {label:<20s}: {med:.2f} +{hi-med:.2f} -{med-lo:.2f}")

---
<a id='sec10'></a>
## 10 · Student exercises

Complete the following exercises to consolidate your understanding.

---

### Exercise 1: Distance--inclination degeneracy

The luminosity distance $d_L$ and inclination angle $\iota$ are degenerate because both affect the observed amplitude.  For a face-on binary ($\iota = 0$) the signal is maximally loud; for an edge-on binary ($\iota = \pi/2$) only the cross-polarisation survives.

**Tasks:**
1. Simulate a 2-D posterior in $(d_L, \iota)$ that shows a banana-shaped degeneracy.
2. Plot the joint posterior as a 2-D histogram or KDE contour.
3. Explain qualitatively why distance can be overestimated when inclination is unconstrained.

---

### Exercise 2: Redshifted vs source-frame chirp mass

For GW150914 at $d_L \approx 440$ Mpc, compute the corresponding redshift $z$ using the Planck 2018 cosmology (`astropy.cosmology.Planck18`).  Then compute:
$$
\mathcal{M}^{\rm source} = \frac{\mathcal{M}^{\rm det}}{1+z}
$$
using the sampled posterior and report the 90% credible interval on both the detector-frame and source-frame chirp masses.

---

### Exercise 3: Spin prior effect on $\chi_{\rm eff}$

Repeat the prior sensitivity analysis from Section 9 but for $\chi_{\rm eff}$:
1. Use an **isotropic spin prior** (component magnitudes uniform in $[0, 1]$, directions isotropic) and derive the induced prior on $\chi_{\rm eff}$.
2. Use a **uniform prior on $\chi_{\rm eff}$** directly.
3. Overlay the resulting posteriors for GW150914.

---

### Exercise 4: Jensen–Shannon divergence between approximants

Using the waveform-model comparison in Section 9, compute the pairwise Jensen–Shannon divergence matrix between all four approximants for the chirp-mass posterior.  Display the result as a heatmap and identify which pair of approximants is most discrepant.

---

### Exercise 5: GW191204_110529 — another BBH event

Choose a BBH event from GWTC-3 (e.g. GW191204_110529) and:
1. Download the GWOSC strain data.
2. Estimate the PSD and plot the ASD.
3. Set up a Bilby prior and run a quick PE job (use `nlive=200`, `maxmcmc=2000`).
4. Compare your $d_L$ and $\mathcal{M}$ posteriors with the published GWTC-3 samples.


In [ ]:
# ---------------------------------------------------------------------------
# Exercise 1 starter: distance-inclination degeneracy
# ---------------------------------------------------------------------------
rng_ex = np.random.default_rng(777)
N_ex   = 8000

# Simulate a (d_L, iota) posterior with a banana-shaped degeneracy.
# The approximate relation is: d_L ~ A / (1 + cos^2(iota))
# where A is set by the SNR.  We model this by sampling iota from
# a sine prior and computing d_L from a Gaussian signal amplitude.

iota_samples  = np.arccos(rng_ex.uniform(-1, 1, N_ex))  # isotropic prior
snr_amplitude = rng_ex.normal(25, 3, N_ex)               # measured SNR

# Approximate amplitude factor (two polarisations add in quadrature)
antenna_factor = np.sqrt(
    ((1 + np.cos(iota_samples)**2) / 2)**2 + np.cos(iota_samples)**2
)
# d_L proportional to amplitude / SNR
amp_0 = 25 * 440  # nominal SNR * nominal d_L
dL_samples = amp_0 * antenna_factor / snr_amplitude

fig, ax = plt.subplots(figsize=(7, 5))
h = ax.hist2d(
    iota_samples * 180 / np.pi, dL_samples,
    bins=[60, 60], range=[[0, 180], [0, 1500]],
    cmap="Blues", density=True
)
plt.colorbar(h[3], ax=ax, label="Probability density")
ax.set_xlabel(r"Inclination $\iota$ [deg]")
ax.set_ylabel(r"$d_L$ [Mpc]")
ax.set_title(r"$d_L$–$\iota$ degeneracy (GW150914-like)")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Face-on (iota~0) : maximum amplitude -> tightest d_L constraint.")
print("  Edge-on (iota~90): minimum amplitude -> d_L poorly constrained / overestimated.")

In [ ]:
# ---------------------------------------------------------------------------
# Exercise 2: Convert detector-frame to source-frame chirp mass
# ---------------------------------------------------------------------------
from astropy.cosmology import Planck18
import astropy.units as u
from astropy.cosmology import z_at_value

dL_posterior_mpc = our_dL_bbh  # GW150914 d_L samples in Mpc
Mc_det_posterior = our_Mc_bbh  # GW150914 Mc samples (detector frame)

# Convert d_L -> z for each sample
print("Converting d_L -> z for GW150914 posterior samples ...")
z_samples = np.array([
    z_at_value(Planck18.luminosity_distance, dL_val * u.Mpc).value
    for dL_val in dL_posterior_mpc[:200]  # subset for speed
])
Mc_src_samples = Mc_det_posterior[:200] / (1 + z_samples)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, xlabel, color in zip(
    axes,
    [Mc_det_posterior[:200], Mc_src_samples],
    [r"$\mathcal{M}^{\rm det}\,[M_\odot]$", r"$\mathcal{M}^{\rm source}\,[M_\odot]$"],
    ["darkorange", "mediumseagreen"]
):
    lo, med, hi = np.percentile(data, [5, 50, 95])
    ax.hist(data, bins=30, density=True, color=color, alpha=0.7)
    ax.axvline(med, color=color, lw=2, ls="-")
    ax.axvline(lo, color=color, lw=1.5, ls="--")
    ax.axvline(hi, color=color, lw=1.5, ls="--")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Probability density")
    ax.set_title(f"{xlabel.split('[')[0]}: {med:.2f} [{lo:.2f}, {hi:.2f}] (90%)")

fig.suptitle("GW150914: Detector-frame vs source-frame chirp mass", fontsize=12)
plt.tight_layout()
plt.show()

med_z = np.median(z_samples)
print(f"Median redshift: z = {med_z:.4f}")
print(f"Chirp mass shift: (1+z) = {1+med_z:.4f}")

---
<a id='sec11'></a>
## 11 · References

### Core papers

1. **Abbott et al. (LVK), 2016** — *Observation of Gravitational Waves from a Binary Black Hole Merger*, PRL 116, 061102. [arXiv:1602.03837](https://arxiv.org/abs/1602.03837)

2. **Abbott et al. (LVK), 2017** — *GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral*, PRL 119, 161101. [arXiv:1710.05832](https://arxiv.org/abs/1710.05832)

3. **Abbott et al. (LVK), 2019** — *GWTC-1: A Gravitational-Wave Transient Catalog of Compact Binary Mergers*, PRX 9, 031040. [arXiv:1811.12907](https://arxiv.org/abs/1811.12907)

4. **Abbott et al. (LVK), 2021** — *GWTC-2: Compact Binary Coalescences Observed by LIGO and Virgo During the First Half of the Third Observing Run*, PRX 11, 021053. [arXiv:2010.14527](https://arxiv.org/abs/2010.14527)

5. **Abbott et al. (LVK), 2023** — *GWTC-3: Compact Binary Coalescences Observed by LIGO and Virgo During the Second Part of the Third Observing Run*, PRX 13, 041039. [arXiv:2111.03606](https://arxiv.org/abs/2111.03606)

### Parameter estimation methodology

6. **Ashton et al., 2019** — *Bilby: A user-friendly Bayesian inference library for gravitational-wave astronomy*, ApJS 241, 27. [arXiv:1811.02042](https://arxiv.org/abs/1811.02042)

7. **Veitch et al., 2015** — *Parameter estimation for compact binaries with ground-based gravitational-wave observations using the LALInference software library*, PRD 91, 042003. [arXiv:1409.7215](https://arxiv.org/abs/1409.7215)

8. **Speagle, 2020** — *dynesty: a dynamic nested sampling package*, MNRAS 493, 3132. [arXiv:1904.02180](https://arxiv.org/abs/1904.02180)

### Waveform models

9. **Husa et al., 2016** — *Frequency-domain gravitational waves from nonprecessing black-hole binaries. I.*, PRD 93, 044006. (IMRPhenomD)

10. **Bohe et al., 2017** — *Improved effective-one-body model of spinning, nonprecessing binary black holes*, PRD 95, 044028. (SEOBNRv4)

### Data and software

11. **GWOSC** — [gwosc.org](https://gwosc.org) — Open data portal for LVK observations.

12. **gwpy** — [gwpy.github.io](https://gwpy.github.io) — Python package for GW data analysis.

13. **PESummary** — [lscsoft.docs.ligo.org/pesummary](https://lscsoft.docs.ligo.org/pesummary) — GW posterior sample summary library.

14. **corner.py** — [corner.readthedocs.io](https://corner.readthedocs.io) — Corner plot visualisation.
